# SQL vs DataFrame API — Two Ways to Solve the Same Problem

**Dataset:** `samples.bakehouse` + `samples.tpch` + `samples.wanderbricks`

**Difficulty:** Hard

**Topics:** `spark.sql()`, `createOrReplaceTempView`, SQL CTEs, SQL window functions, SQL subqueries, `spark.catalog`, `DESCRIBE TABLE`, dynamic SQL, equivalence between SQL and DataFrame API

> Spark SQL and the DataFrame API produce **identical execution plans** — choose whichever is more readable for the task.
> SQL shines for multi-step aggregations and reporting. The DataFrame API shines for parameterised, reusable code.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_bookings     = spark.table("samples.wanderbricks.bookings")

# Register temp views so all tables are queryable via spark.sql
df_transactions.createOrReplaceTempView("transactions")
df_franchises.createOrReplaceTempView("franchises")
df_customers.createOrReplaceTempView("customers")
df_orders.createOrReplaceTempView("orders")
df_customer.createOrReplaceTempView("tpch_customer")
df_bookings.createOrReplaceTempView("bookings")

## Problem 1 — Register Views and Query with SQL

Using `spark.sql()`, write a JOIN query that computes total revenue and transaction count per franchise.
The views `transactions` and `franchises` are already registered in the setup cell.

**Expected output columns:** `franchiseID`, `franchiseName`, `total_revenue`, `transaction_count`

> **Tip:** `spark.sql("SELECT ... FROM transactions t JOIN franchises f ON t.franchiseID = f.franchiseID ...")` — write it like standard SQL.

In [ ]:
# Problem 1 — write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'transaction_count' in cols, "Missing column: transaction_count"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
print(f"Problem 1 passed ✓  ({cnt} franchises)")

## Problem 2 — SQL CTEs for Multi-Step Logic

Use a `spark.sql()` query with **two CTEs** to find the top-selling product per day:

```sql
WITH daily_product AS (
    SELECT DATE(dateTime) AS sale_date, product, SUM(totalPrice) AS daily_revenue
    FROM transactions
    GROUP BY 1, 2
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY sale_date ORDER BY daily_revenue DESC) AS rnk
    FROM daily_product
)
SELECT sale_date, product, daily_revenue FROM ranked WHERE rnk = 1
```

**Expected output columns:** `sale_date`, `product`, `daily_revenue`

In [ ]:
# Problem 2 — write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'sale_date' in cols, "Missing column: sale_date"
assert 'product' in cols, "Missing column: product"
assert 'daily_revenue' in cols, "Missing column: daily_revenue"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# one row per day — days should equal distinct dates
distinct_dates = df_transactions.select(F.to_date('dateTime')).distinct().count()
assert cnt <= distinct_dates, "Should have at most one top product per day"
print(f"Problem 2 passed ✓  ({cnt} days covered)")

## Problem 3 — SQL Window Functions

Write a `spark.sql()` query using `RANK() OVER (PARTITION BY ... ORDER BY ...)` to find the top 3 highest-value transactions per franchise.

```sql
SELECT franchiseID, transactionID, product, totalPrice,
       RANK() OVER (PARTITION BY franchiseID ORDER BY totalPrice DESC) AS rnk
FROM transactions
```
Then filter to `rnk <= 3`.

**Expected output columns:** `franchiseID`, `transactionID`, `product`, `totalPrice`, `rnk`

In [ ]:
# Problem 3 — write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'rnk' in cols, "Missing column: rnk"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
max_rank = result_3.agg(F.max('rnk')).collect()[0][0]
assert max_rank <= 3, f"rnk should be <= 3, got max={max_rank}"
print(f"Problem 3 passed ✓  ({cnt} top-3 transactions)")

## Problem 4 — DataFrame API vs SQL: Prove They're Equivalent

Solve the same problem **both ways** and assert the results match.

**Problem:** Count transactions and compute percentage share per `paymentMethod`.

1. DataFrame API version → `result_4`
2. SQL version → `result_4_sql`
3. Assert both have the same row count and the same `transaction_count` values.

**Expected output columns:** `paymentMethod`, `transaction_count`, `pct_share`

In [ ]:
# Problem 4 — write your solution here
# DataFrame API version
# result_4 = df_transactions.groupBy(...).agg(...).withColumn('pct_share', ...)

# SQL version  
# result_4_sql = spark.sql(""" WITH totals AS (...) SELECT ... """)

# Assign result to: result_4

result_4 = None      # replace — DataFrame API version
result_4_sql = None  # replace — SQL version

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 (DataFrame API) is None"
assert result_4_sql is not None, "result_4_sql (SQL version) is None"
cols = [c.lower() for c in result_4.columns]
assert 'paymentmethod' in cols, "Missing column: paymentMethod"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert 'pct_share' in cols, "Missing column: pct_share"
cnt_api = result_4.count()
cnt_sql = result_4_sql.count()
assert cnt_api == cnt_sql, f"Row counts differ: DataFrame API={cnt_api}, SQL={cnt_sql}"
# pct_share should sum to ~100
total_pct = result_4.agg(F.sum('pct_share')).collect()[0][0]
assert abs(total_pct - 100.0) < 1.0, f"pct_share should sum to ~100, got {total_pct}"
print(f"Problem 4 passed ✓  (both methods give {cnt_api} rows, pct sums to {total_pct:.1f}%)")

## Problem 5 — SQL Subquery Pattern (IN / EXISTS)

Find all TPC-H orders where the customer belongs to the `BUILDING` market segment.

Write it as a SQL query using an `IN` subquery:
```sql
SELECT o_orderkey, o_custkey, o_totalprice, o_orderdate
FROM orders
WHERE o_custkey IN (
    SELECT c_custkey FROM tpch_customer WHERE c_mktsegment = 'BUILDING'
)
```

**Expected output columns:** `o_orderkey`, `o_custkey`, `o_totalprice`, `o_orderdate`

In [ ]:
# Problem 5 — write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'o_orderkey' in cols, "Missing column: o_orderkey"
assert 'o_totalprice' in cols, "Missing column: o_totalprice"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
# Verify using DataFrame API semi-join as ground truth
building_customers = df_customer.filter(F.col('c_mktsegment') == 'BUILDING')
expected = df_orders.join(F.broadcast(building_customers), df_orders.o_custkey == building_customers.c_custkey, 'left_semi').count()
assert cnt == expected, f"Expected {expected} rows, got {cnt}"
print(f"Problem 5 passed ✓  ({cnt} BUILDING segment orders)")

## Problem 6 — Explore the Catalog with `spark.catalog`

Use `spark.sql('SHOW TABLES IN samples.bakehouse')` or `spark.catalog` methods to list available tables.
Build a result DataFrame that shows which tables exist in `samples.bakehouse`.

**Expected output columns:** `database`, `tableName`, `isTemporary`

> `spark.catalog.listTables(dbName)` returns a list of `Table` objects. You can also use:
> `spark.sql('SHOW TABLES IN samples.bakehouse')` which returns a DataFrame directly.

In [ ]:
# Problem 6 — write your solution here
# Option A: spark.sql("SHOW TABLES IN samples.bakehouse")
# Option B: spark.createDataFrame([(t.database, t.name, t.isTemporary) for t in spark.catalog.listTables('samples.bakehouse')], ['database','tableName','isTemporary'])
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'tablename' in cols or 'table_name' in cols, "Missing column: tableName or table_name"
cnt = result_6.count()
assert cnt >= 4, f"samples.bakehouse has at least 4 tables, got {cnt}"
print(f"Problem 6 passed ✓  ({cnt} tables found in samples.bakehouse)")

## Problem 7 — `DESCRIBE TABLE` for Schema Metadata

Use `spark.sql('DESCRIBE TABLE samples.bakehouse.sales_transactions')` to retrieve column metadata.
This is useful for data cataloguing, schema documentation, and automated validation.

**Expected output columns:** `col_name`, `data_type`, `comment`

In [ ]:
# Problem 7 — write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'col_name' in cols, "Missing column: col_name"
assert 'data_type' in cols, "Missing column: data_type"
cnt = result_7.count()
assert cnt >= 5, f"Expected at least 5 columns described, got {cnt}"
col_names = [r['col_name'] for r in result_7.filter(F.col('col_name') != '').select('col_name').collect()]
assert 'transactionID' in col_names or 'transactionid' in [c.lower() for c in col_names], "transactionID should appear in description"
print(f"Problem 7 passed ✓  ({cnt} metadata rows)")

## Problem 8 — Dynamic SQL Generation

One power of `spark.sql()` is that the SQL string is just a Python string — you can build it dynamically.

Write a Python function `group_and_count(table_name, group_cols)` that:
1. Takes a registered view name and a list of column names
2. Builds a SQL string: `SELECT {cols}, COUNT(*) AS row_count FROM {table} GROUP BY {cols} ORDER BY row_count DESC`
3. Executes it with `spark.sql()`

Call it with `group_and_count('transactions', ['product', 'paymentMethod'])`.
Assign the result to `result_8`.

**Expected output columns:** `product`, `paymentMethod`, `row_count`

In [ ]:
# Problem 8 — write your solution here
# def group_and_count(table_name, group_cols):
#     cols_str = ', '.join(group_cols)
#     query = f"SELECT {cols_str}, COUNT(*) AS row_count FROM {table_name} GROUP BY {cols_str} ORDER BY row_count DESC"
#     return spark.sql(query)
#
# result_8 = group_and_count('transactions', ['product', 'paymentMethod'])
# Assign result to: result_8

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
assert 'product' in cols, "Missing column: product"
assert 'paymentmethod' in cols, "Missing column: paymentMethod"
assert 'row_count' in cols, "Missing column: row_count"
cnt = result_8.count()
assert cnt > 0, "Got 0 rows"
min_count = result_8.agg(F.min('row_count')).collect()[0][0]
assert min_count >= 1, "row_count must be >= 1"
print(f"Problem 8 passed ✓  ({cnt} product-payment combinations)")